# PA2 stage 05: evaluation generations and blind pair files (Task 6)

SFT vs DPO-UltraFeedback on the fixed AlpacaEval subset (300 prompts, seed 42), and SFT vs
DPO-HH on the provided helpful/harmless set (300 prompts). Same prompt format and decoding for
every model: greedy, `--batch_size 32 --max_new_tokens 512 --repetition_penalty 1.1`. Generation resumes per row; pair files are written before any judging.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --format="%h %s"

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib 
!python -m pip install -q -e . --no-deps

In [ ]:
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2"

In [ ]:
# Let PyTorch grow memory segments instead of fragmenting (inherited by every ! and subprocess call).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Link the attached prepared-data dataset to data/prepared, wherever Kaggle mounted it.
import glob
hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert hits, "attach the pa2-alignment-data dataset"
DATA_DIR = os.path.dirname(hits[0])
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/prepared"):
    os.symlink(DATA_DIR, "data/prepared")
!ls -la data/prepared/
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
import json

def find_ckpt(run):
    hits = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}/best/config.json", recursive=True))
    assert hits, f"attach the stage output that contains outputs/{run}/best"
    return os.path.dirname(hits[-1])

CKPT = {"sft": find_ckpt("sft"), "dpo_hh": find_ckpt("dpo_hh"), "dpo_uf": find_ckpt("dpo_uf")}
print(json.dumps(CKPT, indent=2))

In [ ]:
# Resume support: copy this stage's run folders back from a previous run's attached output.
import shutil
for run in ['gen_eval']:
    local = os.path.join("outputs", run)
    if os.path.exists(local):
        continue
    found = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}", recursive=True))
    if found:
        print(f"restoring outputs/{run} from {found[-1]}")
        shutil.copytree(found[-1], local, symlinks=False)
    else:
        print(f"outputs/{run}: starting fresh")
!du -sh outputs/* 2>/dev/null || true

In [ ]:
# GPU 0: SFT on both sets. GPU 1: each DPO model on its own set. Two processes in parallel.
jobs = {
    "0": [("sft", "alpaca_eval_300_seed42"), ("sft", "hh_rlhf_eval_300_seed42")],
    "1": [("dpo_uf", "alpaca_eval_300_seed42"), ("dpo_hh", "hh_rlhf_eval_300_seed42")],
}
os.makedirs("outputs/gen_eval", exist_ok=True)
procs = []
for gpu, runs in jobs.items():
    cmd = " && ".join(
        f"python -u scripts/generate.py --model {CKPT[model]} --generator qwen2.5-0.5b-{model} "
        f"--eval_set eval_sets/{evalset}.json --out outputs/gen_eval/{evalset}__{model}.json --batch_size 32 --max_new_tokens 512 --repetition_penalty 1.1"
        for model, evalset in runs)
    procs.append(subprocess.Popen(cmd, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONPATH": "."},
                                  stdout=open(f"outputs/gen_eval/gpu{gpu}.log", "a"), stderr=subprocess.STDOUT))
print("exit codes", [p.wait() for p in procs])
!tail -n 3 outputs/gen_eval/gpu0.log outputs/gen_eval/gpu1.log

In [ ]:
!PYTHONPATH=. python scripts/build_pairs.py --a outputs/gen_eval/alpaca_eval_300_seed42__sft.json --a_name sft \
    --b outputs/gen_eval/alpaca_eval_300_seed42__dpo_uf.json --b_name dpo_uf --out outputs/gen_eval/pairs_alpaca_dpo_uf_vs_sft.jsonl
!PYTHONPATH=. python scripts/build_pairs.py --a outputs/gen_eval/hh_rlhf_eval_300_seed42__sft.json --a_name sft \
    --b outputs/gen_eval/hh_rlhf_eval_300_seed42__dpo_hh.json --b_name dpo_hh --out outputs/gen_eval/pairs_hh_dpo_hh_vs_sft.jsonl
for f in sorted(glob.glob("outputs/gen_eval/*__*.json")):
    if f.endswith(".settings.json"):
        continue
    rows = json.load(open(f))
    n_len = sum(r["finish_reason"] == "length" for r in rows)
    mean_words = sum(len(r["output"].split()) for r in rows) / len(rows)
    print(f"{os.path.basename(f):45s} rows={len(rows)} hit_max_tokens={n_len} mean_words={mean_words:.0f}")

In [ ]:
!cd outputs && zip -qr /kaggle/working/gen_eval_outputs.zip gen_eval
!du -sh outputs/gen_eval /kaggle/working/gen_eval_outputs.zip